In [5]:
import numpy as np
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
from scipy.sparse import lil_matrix, csr_matrix

In [6]:
# Download necessary NLTK data (if you haven't already)
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

In [8]:
# --- 1. Tokenize the text ---
with open('text8', 'r') as f:
    text = f.read()

# Tokenize, lowercase, and remove stopwords and non-alphabetic characters
stop_words = set(stopwords.words('english'))
tokens = [word.lower() for word in word_tokenize(text) if word.isalpha() and word.lower() not in stop_words]

# --- Build a vocabulary of the 20,000 most common words ---
vocab_size = 20000
word_counts = Counter(tokens)
vocab = [word for word, count in word_counts.most_common(vocab_size)]
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

In [9]:
# --- 2. Compute a sparse word co-occurrence matrix ---
window_size = 2
co_occurrence_matrix = lil_matrix((vocab_size, vocab_size), dtype=np.float64)

for i, token in enumerate(tokens):
    if token in word_to_idx:
        for j in range(max(0, i - window_size), min(len(tokens), i + window_size + 1)):
            if i != j and tokens[j] in word_to_idx:
                co_occurrence_matrix[word_to_idx[token], word_to_idx[tokens[j]]] += 1

In [10]:
# --- 3. Apply Positive Pointwise Mutual Information (PPMI) ---
def ppmi(co_occurrence_matrix):
    total_co_occurrences = co_occurrence_matrix.sum()
    word_sums = np.array(co_occurrence_matrix.sum(axis=1)).flatten()
    context_sums = np.array(co_occurrence_matrix.sum(axis=0)).flatten()

    # Convert to coordinate format for efficient iteration
    coo_matrix = co_occurrence_matrix.tocoo()
    
    ppmi_rows, ppmi_cols, ppmi_data = [], [], []

    for i, j, v in zip(coo_matrix.row, coo_matrix.col, coo_matrix.data):
        if v > 0:
            p_i_j = v / total_co_occurrences
            p_i = word_sums[i] / total_co_occurrences
            p_j = context_sums[j] / total_co_occurrences
            
            pmi = np.log2(p_i_j / (p_i * p_j))
            if pmi > 0:
                ppmi_rows.append(i)
                ppmi_cols.append(j)
                ppmi_data.append(pmi)
                
    return csr_matrix((ppmi_data, (ppmi_rows, ppmi_cols)), shape=co_occurrence_matrix.shape)


# Convert to CSR format which is better for arithmetic operations
co_occurrence_matrix_csr = co_occurrence_matrix.tocsr()
ppmi_matrix = ppmi(co_occurrence_matrix_csr)

In [11]:
# --- 4. Reduce dimensionality using Truncated SVD ---
n_components = 100
svd = TruncatedSVD(n_components=n_components)
word_embeddings = svd.fit_transform(ppmi_matrix)

In [12]:
# --- 5. Find the 10 most similar words ---
target_word = "exceptional"
if target_word in word_to_idx:
    target_word_embedding = word_embeddings[word_to_idx[target_word]]
    
    similarities = cosine_similarity([target_word_embedding], word_embeddings)[0]
    most_similar_indices = similarities.argsort()[-11:-1][::-1]
    
    print(f"The 10 most similar words to '{target_word}' are:")
    for idx in most_similar_indices:
        print(idx_to_word[idx])
else:
    print(f"'{target_word}' is not in the vocabulary.")

The 10 most similar words to 'exceptional' are:
extraordinary
remarkable
unusual
subtle
stature
possessing
mastery
desirable
importantly
clarity


In [13]:
# --- 6. Visualize selected words in 3D space using PCA ---
words_to_visualize = ["king", "queen", "man", "woman", "france", "germany", "paris", "berlin", "apple", "microsoft"]
# Filter for words that are actually in our vocabulary
words_to_visualize = [word for word in words_to_visualize if word in word_to_idx]

word_indices_to_visualize = [word_to_idx[word] for word in words_to_visualize]
embeddings_to_visualize = word_embeddings[word_indices_to_visualize]

pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(embeddings_to_visualize)

print("\n3D coordinates for visualization:")
for i, word in enumerate(words_to_visualize):
    print(f"{word}: {embeddings_3d[i]}")


3D coordinates for visualization:
king: [37.6346345  -9.41795828 35.62831183]
queen: [ 20.86261753 -13.43880946  19.88761287]
man: [ 13.44514434 -39.41755365 -23.69568774]
woman: [  6.4266631  -33.48419057 -21.15740818]
france: [16.64412854 25.73930802  6.43725152]
germany: [ 7.2503063  37.71881789 -6.70395367]
paris: [  5.3902823   17.95469933 -17.66728504]
berlin: [ -3.08274107  27.64865383 -17.52760236]
apple: [-47.56943281  -9.72656009  10.07491419]
microsoft: [-57.00160275  -3.57640702  14.72384659]
